<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
from src.patterns.scanner import scan_all_patterns
from src.data.utils import compute_atr
from src.features.build_features import compute_all_indicators

def bars_since_last_event(event_mask, max_val=10):
    event_mask = event_mask.values if hasattr(event_mask, 'values') else event_mask
    result = np.full(len(event_mask), max_val, dtype=int)
    last_seen = -1
    for i in range(len(event_mask)):
        if event_mask[i]:
            last_seen = i
            result[i] = 0
        elif last_seen >= 0:
            result[i] = min(i - last_seen, max_val)
    return result

def label_every_bar(df, pt_mult=2.0, sl_mult=2.0, max_holding=10, atr_window=14):
    atr = compute_atr(df, window=atr_window)
    results = []
    for i in range(len(df) - max_holding):
        entry_price = df["Close"].iloc[i]
        atr_val = atr.iloc[i]
        if pd.isna(atr_val) or atr_val <= 0:
            continue
        upper = entry_price + pt_mult * atr_val
        lower = entry_price - sl_mult * atr_val
        label = "no_trade"
        exit_price = df["Close"].iloc[i + max_holding]
        exit_date = df.index[i + max_holding]
        bars_held = max_holding
        for j in range(i + 1, i + max_holding + 1):
            high_j = df["High"].iloc[j]
            low_j = df["Low"].iloc[j]
            hit_upper = high_j >= upper
            hit_lower = low_j <= lower
            if hit_upper and hit_lower:
                exit_price = df["Close"].iloc[j]; exit_date = df.index[j]; bars_held = j - i; break
            elif hit_upper:
                label = "long"; exit_price = upper; exit_date = df.index[j]; bars_held = j - i; break
            elif hit_lower:
                label = "short"; exit_price = lower; exit_date = df.index[j]; bars_held = j - i; break
        results.append({"event_date": df.index[i], "label": label})
    return pd.DataFrame(results)

def build_bar_by_bar_enriched(ticker, start='2010-01-01', end='2026-01-01'):
    """Full pipeline: download, scan, label every bar, attach indicators + recency features."""
    raw = yf.download(ticker, start=start, end=end, auto_adjust=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(1)
    raw = raw[['Open', 'High', 'Low', 'Close', 'Volume']]
    if raw.index.tz is not None:
        raw.index = raw.index.tz_localize(None)
    raw.index.name = 'Date'

    scanned = scan_all_patterns(raw.copy())
    labeled = label_every_bar(raw.copy())
    indicators = compute_all_indicators(raw.copy())

    labeled_valid = labeled[labeled['event_date'].isin(indicators.index)].reset_index(drop=True)
    dates = pd.DatetimeIndex(labeled_valid['event_date'])

    feats = indicators.loc[dates].reset_index(drop=True)
    abs_sma_cols = [c for c in feats.columns if c.startswith("sma_") and "_dist" not in c]
    feats = feats.drop(columns=abs_sma_cols, errors="ignore")
    feats = feats.dropna(axis=1, how="all")

    recency = pd.DataFrame({
        'bars_since_near_support': bars_since_last_event(scanned['near_support']),
        'bars_since_near_resistance': bars_since_last_event(scanned['near_resistance']),
        'bars_since_triangle': bars_since_last_event(scanned['triangle_pattern'].notna()),
        'bars_since_channel': bars_since_last_event(scanned['channel_pattern'].notna()),
        'bars_since_mtb': bars_since_last_event(scanned['multiple_top_bottom_pattern'].notna()),
    }, index=scanned.index).loc[dates].reset_index(drop=True)

    feats_enriched = pd.concat([feats, recency], axis=1)
    labels = labeled_valid['label']

    return feats_enriched, labels

TICKERS = ['SPY', 'QQQ', 'NVDA', 'GOOGL', 'AAPL', 'MSFT', 'AMZN', 'META', 'JPM', 'XOM']

pooled_bar_data = {}
for t in TICKERS:
    feats, labs = build_bar_by_bar_enriched(t)
    pooled_bar_data[t] = {'features': feats, 'labels': labs}
    print(f"{t}: {len(feats)} bars labeled")